# GraphRAG — Part 1: Ingestion (Indexing)

This notebook teaches the **ingestion / indexing** half of Microsoft's [GraphRAG](https://github.com/microsoft/graphrag)
package: turning a folder of plain-text documents into a queryable knowledge graph.

**What GraphRAG's indexing pipeline does, in one paragraph:** 
It chunks your documents, uses an LLM to extract
entities and relationships from each chunk, builds a graph out of them, detects *communities* (clusters of
related entities) using the Leiden algorithm, and asks the LLM to summarize each community into a
human-readable report. Those entities, relationships, and community reports are what Part 2 (retrieval) queries.

### Prerequisites
- **Python 3.10, 3.11, or 3.12.** The `graphrag` package does not currently support 3.13+. 
- An **OpenAI API key** with access to a chat model (e.g. `gpt-4o-mini`) and an embedding model
  (e.g. `text-embedding-3-small`).

### ⚠️ Cost warning
Indexing calls the LLM once per chunk (extraction) plus once per detected community (summarization). This is
**not free**, and cost scales with corpus size. 

Start with a handful of small `.txt` files to learn the workflow before pointing this at anything large.


In [3]:
from pathlib import Path
import shutil
import sys

INPUT_DATA_DIR = "my_text_files"

LOCAL_BASE_DIR = Path(".")
PROJECT_ROOT = LOCAL_BASE_DIR / "graphrag_project"

SOURCE_DOCS_DIR = LOCAL_BASE_DIR / INPUT_DATA_DIR

SOURCE_DOCS_DIR.mkdir(parents=True, exist_ok=True)
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

print(f"Project root (live) : {PROJECT_ROOT.resolve()}")
print(f"Source docs dir     : {SOURCE_DOCS_DIR.resolve()}")

# List all files inside the source directory
files = [f.name for f in SOURCE_DOCS_DIR.iterdir() if f.is_file()]

print(f"Files in {SOURCE_DOCS_DIR.name}:")
print(files)

Project root (live) : C:\Users\hi\Desktop\projects\python_projects\tutorial\play_langchain_llamaindex_langgraph\graphrag_project
Source docs dir     : C:\Users\hi\Desktop\projects\python_projects\tutorial\play_langchain_llamaindex_langgraph\my_text_files
Files in my_text_files:
['01_company.txt']


## Step 2 — Install dependencies

`graphrag` is a separate package from your existing `langchain`/`llama-index` stack — it does not conflict with
them, but it does pull in its own dependencies (pandas, pyarrow, networkx, tiktoken, LiteLLM, etc.), so give the
install a minute.

We also install `langchain-community`, which provides the `DirectoryLoader`/`TextLoader` combo used below to read
every `.txt` file in a directory — this is the same file-reading approach you're likely already using elsewhere,
rather than hand-rolling `os.walk` + `open()`.

> Note: `langchain-community` was officially sunset in mid-2026 (no new features, community-maintained only).
> It still works fine for this use case today, but if you're starting something new and long-lived, keep an eye on
> LangChain's per-provider standalone packages as the longer-term direction.


In [2]:
# Run following in command line
# python -m pip install -q graphrag  pyyaml 
# %pip install langchain-community

# %pip install -q python-dotenv

^C
Note: you may need to restart the kernel to use updated packages.


In [2]:
import importlib.metadata

# List the distribution package names
packages = ["graphrag", "pyyaml", "langchain-community"]

for package in packages:
    try:
        version = importlib.metadata.version(package)
        print(f"{package} version: {version}")
    except importlib.metadata.PackageNotFoundError:
        print(f"{package} is not installed in this environment.")


graphrag version: 3.1.1
pyyaml version: 6.0.3
langchain-community version: 0.4.2


## Step 3 — Read every text file in the directory

We use LangChain's `DirectoryLoader` (backed by `TextLoader`) to load every `.txt` file under `SOURCE_DOCS_DIR`,
including subdirectories (`glob="**/*.txt"`). Each loaded item is a LangChain `Document` with `.page_content`
(the text) and `.metadata["source"]` (the original file path) — this is the same loader pattern used for
standard RAG ingestion, so if you already have a directory-loading step elsewhere in your stack, this slots in
the same way.


In [3]:
from langchain_community.document_loaders import DirectoryLoader, TextLoader

loader = DirectoryLoader(
    str(SOURCE_DOCS_DIR),
    glob="**/*.txt",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    show_progress=True,
)

documents = loader.load()
print(f"Loaded {len(documents)} document(s) from {SOURCE_DOCS_DIR}")

for doc in documents[:3]:
    print("-", doc.metadata["source"], f"({len(doc.page_content)} chars)")


C:\Users\hi\AppData\Local\Temp\ipykernel_368\3801252578.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import DirectoryLoader, TextLoader
100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:00<00:00, 55.54it/s]

Loaded 1 document(s) from my_text_files
- my_text_files\01_company.txt (1652 chars)


## Step 4 — Initialize the GraphRAG project

`graphrag init` scaffolds the project:
`settings.yaml` (pipeline config),
`.env` (API key),
`input/` (where GraphRAG reads documents from), and
`prompts/` (default LLM prompts for extraction/summarization, which you can
customize later via `graphrag prompt-tune`).

Passing `--model`/`--embedding` here sets your chat and embedding models directly and non-interactively — no
prompts to answer. `--force` lets you re-run this cell safely (it overwrites `settings.yaml`/`prompts/`, not
your `input/` documents).


In [4]:
import subprocess

MODEL = "gpt-4o-mini"
EMBEDDING = "text-embedding-3-small"

result = subprocess.run(
    [
        "graphrag", "init",
        "--root", str(PROJECT_ROOT),
        "--model", MODEL,
        "--embedding", EMBEDDING,
        "--force",
    ],
    capture_output=True, text=True,
)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
    raise RuntimeError("graphrag init failed — see stderr above")


## Step 5 — Copy the loaded documents into GraphRAG's `input/` folder

GraphRAG indexes whatever `.txt` files it finds in `<project_root>/input/`. We write the documents we loaded in
Step 3 there, using a sanitized version of the original filename so you can trace an entity back to its source
document later.


In [5]:
import re

input_dir = PROJECT_ROOT / "input"
input_dir.mkdir(parents=True, exist_ok=True)

def safe_filename(source_path: str, index: int) -> str:
    stem = Path(source_path).stem
    stem = re.sub(r"[^A-Za-z0-9_-]+", "_", stem) or f"doc_{index}"
    return f"{stem}.txt"

for i, doc in enumerate(documents):
    out_path = input_dir / safe_filename(doc.metadata["source"], i)
    out_path.write_text(doc.page_content, encoding="utf-8")

written = sorted(input_dir.glob("*.txt"))
print(f"Wrote {len(written)} file(s) to {input_dir}")
for p in written[:5]:
    print("-", p.name)


Wrote 1 file(s) to graphrag_project\input
- 01_company.txt


## Step 6 — Set your OpenAI API key

`graphrag init` created a `.env` file with a placeholder: `GRAPHRAG_API_KEY=<API_KEY>`. `settings.yaml`
references this variable (as `${GRAPHRAG_API_KEY}`) for both the chat and embedding model, so one key covers
both. `getpass` is used here so your key doesn't get echoed into notebook output or saved in cell history.


In [6]:
import getpass
import os

api_key = os.environ.get("OPENAI_API_KEY") or getpass.getpass("Enter your OpenAI API key: ")

env_path = PROJECT_ROOT / ".env"
env_path.write_text(f"GRAPHRAG_API_KEY={api_key}\n", encoding="utf-8")

print(f"Wrote API key to {env_path}")


Enter your OpenAI API key:  ········


Wrote API key to graphrag_project\.env


## Step 7 — What's actually happening during extraction

It's easy to think of `graphrag index` as a black box, so before running it: entity/relationship extraction
*is* a plain prompt-based LLM call under the hood — GraphRAG just generated and wired it up for you in Step 4,
rather than you writing it yourself (the same idea as a custom extraction prompt, just handled internally).

Three things control it, all in `settings.yaml`:
- **`extract_graph.entity_types`** — the categories the LLM is asked to look for (defaults to
  `[organization, person, geo, event]`). Edit this list to fit your domain — e.g. add `product`, `technology`.
- **`extract_graph.max_gleanings`** — after the first extraction pass over a chunk, the LLM is asked *"did you
  miss anything?"* and does this many additional passes if so. Higher = better recall, more LLM calls.
- **`extract_graph.prompt`** — the actual prompt file. Let's look at it.


In [8]:
entity_types_line = [l for l in (PROJECT_ROOT / "settings.yaml").read_text().splitlines()
                     if l.strip().startswith("entity_types")][0]
print(entity_types_line)
print()

extraction_prompt = (PROJECT_ROOT / "prompts" / "extract_graph.txt").read_text(encoding="utf-8")
print(extraction_prompt)

  entity_types: [organization,person,geo,event]


-Goal-
Given a text document that is potentially relevant to this activity and a list of entity types, identify all entities of those types from the text and all relationships among the identified entities.
 
-Steps-
1. Identify all entities. For each identified entity, extract the following information:
- entity_name: Name of the entity, capitalized
- entity_type: One of the following types: [{entity_types}]
- entity_description: Comprehensive description of the entity's attributes and activities
Format each entity as ("entity"<|><entity_name><|><entity_type><|><entity_description>)
 
2. From the entities identified in step 1, identify all pairs of (source_entity, target_entity) that are *clearly related* to each other.
For each pair of related entities, extract the following information:
- source_entity: name of the source entity, as identified in step 1
- target_entity: name of the target entity, as identified in step 1
- relationshi

Notice the output format the prompt asks for: `("entity"<|>NAME<|>TYPE<|>DESCRIPTION)` and
`("relationship"<|>SOURCE<|>TARGET<|>DESCRIPTION<|>STRENGTH)`, separated by `##`. GraphRAG parses this delimited
text back into structured rows — an older, more token-efficient format than JSON/function-calling, chosen
because this prompt runs once per chunk (and again for every gleaning pass), so token efficiency compounds.

If you want the prompt tailored to your own corpus instead of the generic examples shown above (finance,
geopolitics), run `graphrag prompt-tune --root <project_root>` — it samples your actual documents and
regenerates `extract_graph.txt` (and the summarization/community-report prompts) with domain-relevant examples.
You can also just edit the prompt file directly, same as you would in a hand-rolled extraction pipeline.

For reference, `settings.yaml` also controls chunking (further up in the file):


In [9]:
settings_text = (PROJECT_ROOT / "settings.yaml").read_text(encoding="utf-8")

print(settings_text[: settings_text.index("### Storage settings")])

### This config file contains required core defaults that must be set, along with a handful of common optional settings.
### For a full list of available settings, see https://microsoft.github.io/graphrag/config/yaml/

### LLM settings ###
## There are a number of settings to tune the threading and token limits for LLM calls - check the docs.

completion_models:
  default_completion_model:
    model_provider: openai
    model: gpt-4o-mini
    auth_method: api_key # or azure_managed_identity
    api_key: ${GRAPHRAG_API_KEY} # set this in the generated .env file, or remove if managed identity
    retry:
      type: exponential_backoff

embedding_models:
  default_embedding_model:
    model_provider: openai
    model: text-embedding-3-small
    auth_method: api_key
    api_key: ${GRAPHRAG_API_KEY}
    retry:
      type: exponential_backoff

### Document processing settings ###

input:
  type: text # [csv, text, json, jsonl]

chunking:
  type: tokens
  size: 1200
  overlap: 100
  encoding_

## Step 8 — Run the indexing pipeline

This is the step that calls the LLM: 

chunking → entity/relationship extraction → community detection →
community report summarization → embeddings. 

It streams progress live. For a handful of small files this
usually takes a few minutes; scale accordingly for larger corpora.

Re-running this cell is cached by default (`--cache`, the default) — unchanged chunks won't be re-sent to the
LLM. Use `--no-cache` to force a full re-run (e.g. after changing the extraction prompt).


In [10]:
# This took 3 minutes
import subprocess

process = subprocess.Popen(
    ["graphrag", "index", "--root", str(PROJECT_ROOT)],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)

for line in process.stdout:
    print(line, end="")

process.wait()
if process.returncode != 0:
    raise RuntimeError(f"graphrag index exited with code {process.returncode}")


Starting pipeline with workflows: load_input_documents, create_base_text_units, create_final_documents, extract_graph, finalize_graph, extract_covariates, create_communities, create_final_text_units, create_community_reports, generate_text_embeddings
Starting workflow: load_input_documents

Workflow complete: load_input_documents
Starting workflow: create_base_text_units
  1 / 1 ............................................................................................

Workflow complete: create_base_text_units
Starting workflow: create_final_documents

Workflow complete: create_final_documents
Starting workflow: extract_graph
  1 / 1 ............................................................................................
  1 / 1 ............................................................................................
  1 / 20 
  2 / 20 .
  3 / 20 ......
  4 / 20 ...........
  5 / 20 ................
  6 / 20 .....................
  7 / 20 ..........................
  8 / 20 ..

## Step 9 — Inspect the output artifacts

Once indexing completes, `<project_root>/output/` contains a set of Parquet files — this is the knowledge graph,
ready for Part 2 (retrieval).


In [11]:
output_dir = PROJECT_ROOT / "output"
parquet_files = sorted(output_dir.glob("*.parquet"))
print("Output artifacts:")

for p in parquet_files:
    print("-", p.name)

Output artifacts:
- communities.parquet
- community_reports.parquet
- documents.parquet
- entities.parquet
- relationships.parquet
- text_units.parquet


In [13]:
import pandas as pd

entities = pd.read_parquet(output_dir / "entities.parquet")
relationships = pd.read_parquet(output_dir / "relationships.parquet")
community_reports = pd.read_parquet(output_dir / "community_reports.parquet")
text_units = pd.read_parquet(output_dir / "text_units.parquet")

print(f"Entities: {len(entities)}, Relationships: {len(relationships)}, Community reports: {len(community_reports)}")
entities[["title", "type", "description"]].head(20)


Entities: 10, Relationships: 10, Community reports: 3


,title,type,description
0,MERIDIAN HEALTH SYSTEMS,ORGANIZATION,Meridian Health Systems is a hospital network ...
1,NOVACARE ROBOTICS,ORGANIZATION,NovaCare Robotics is a company founded in 2018...
2,DR. AMARA OSEI,PERSON,Dr. Amara Osei is the Chief Innovation Officer...
3,WEI ZHANG,PERSON,Wei Zhang is the engineer who founded NovaCare...
4,TEXAS NURSES COALITION,ORGANIZATION,The Texas Nurses Coalition is a group that rai...
5,CEDAR RIDGE MEDICAL,ORGANIZATION,Cedar Ridge Medical is a hospital that signed ...
6,NATIONAL HEALTHCARE INNOVATION SUMMIT,EVENT,The National Healthcare Innovation Summit is a...
7,AUSTIN,GEO,Austin is the city in Texas where Meridian Hea...
8,2024,EVENT,March 2024 is when Meridian Health Systems ann...
9,2025,EVENT,"By the end of 2025, NovaCare Robotics emerged ..."


### Tracing an entity back to the exact prompt call that produced it

To make the extraction mechanism from Step 7 concrete: every entity records which text chunk(s)
(`text_unit_ids`) it was extracted from. Here's the full chain, end to end, for one entity.


In [14]:
sample_entity = entities.iloc[0]

print(f"Entity   : {sample_entity['title']}  ({sample_entity['type']})")
print(f"Description: {sample_entity['description']}\n")

matching_units = text_units[text_units["id"].isin(sample_entity["text_unit_ids"])]
print("Source text chunk(s) the LLM was actually shown when it extracted this entity:")
for _, unit in matching_units.iterrows():
    print("-", unit["text"])


Entity   : MERIDIAN HEALTH SYSTEMS  (ORGANIZATION)
Description: Meridian Health Systems is a hospital network based in Austin, Texas that has partnered to deploy surgical assistance robots across its twelve facilities.

Source text chunk(s) the LLM was actually shown when it extracted this entity:
- Meridian Health Systems, a hospital network based in Austin, Texas, announced a partnership with NovaCare Robotics in March 2024 to deploy surgical assistance robots across its twelve facilities. The deal was negotiated by Meridian's Chief Innovation Officer, Dr. Amara Osei, who had previously led a similar robotics initiative at Cedar Ridge Medical before joining Meridian in 2022. NovaCare Robotics, founded in 2018 by engineer Wei Zhang, had built its reputation supplying robotic systems to research hospitals in Boston before expanding into the broader commercial healthcare market.

The partnership faced early resistance from the Texas Nurses Coalition, which raised concerns about job disp

## Next steps

Your knowledge graph is built. Open **`02_graphrag_retrieval.ipynb`** — it uses the same restore-from-Drive-backup
logic as Step 1 here, so if you're starting a fresh Colab session it will pull the project back down
automatically before querying it.

To re-index after adding more documents to `input/`, you can either re-run Step 8 (full re-index, using the
cache for unchanged chunks) or use `graphrag update --root <project_root>` for an incremental update — worth
knowing about, not covered in depth here. Re-run Step 10 afterward to refresh the Drive backup.
